# Direction 3 Multi-Dataset Colab Runner

This version is restart-safe for free-tier Colab.

What changed:
- the repository lives inside Google Drive instead of `/content`
- all generated models, CSVs, PNGs, and logs persist in Drive automatically
- each dataset also gets its own zip export in Drive after the run finishes
- you can reconnect later and continue from the same Drive-backed workspace

Recommended runtime: `T4 GPU`.
Do not run the training cells on CPU unless you intentionally want a very slow run.


In [ ]:
from google.colab import drive
from pathlib import Path

DRIVE_MOUNT = Path('/content/drive')
drive.mount(str(DRIVE_MOUNT))

WORKSPACE_ROOT = DRIVE_MOUNT / 'MyDrive' / 'direction3_multidataset_workspace'
REPO_DIR = WORKSPACE_ROOT / 'synthetic-data-lifecycle-benchmarks'
EXPORT_DIR = WORKSPACE_ROOT / 'exports'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Workspace root:', WORKSPACE_ROOT)
print('Repo dir:', REPO_DIR)
print('Export dir:', EXPORT_DIR)


In [ ]:
import subprocess
from pathlib import Path
import os

if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/Shubhamisl/synthetic-data-lifecycle-benchmarks.git', str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', 'main'], check=True)

os.chdir(REPO_DIR)
print('Current working directory:', Path.cwd())


In [ ]:
%cd /content/drive/MyDrive/direction3_multidataset_workspace/synthetic-data-lifecycle-benchmarks
!pip install -q -r requirements_colab_direction3.txt


In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU runtime not available. Reconnect to a T4 GPU before running training cells.')


In [ ]:
%cd /content/drive/MyDrive/direction3_multidataset_workspace/synthetic-data-lifecycle-benchmarks
!python -m data.loader
!python -m benchmarks.download_datasets


## Optional Dry-Runs

Use these only to validate wiring. They do not train models.


In [ ]:
%cd /content/drive/MyDrive/direction3_multidataset_workspace/synthetic-data-lifecycle-benchmarks
!python -m dp_triangle.run_direction3 --dataset adult --dry-run
!python -m dp_triangle.run_direction3 --dataset bank --dry-run
!python -m dp_triangle.run_direction3 --dataset diabetes --dry-run
!python -m dp_triangle.run_direction3 --dataset covertype --dry-run


## Archiving Helpers

These helpers write zip exports into Drive after each dataset run. Even if Colab resets later, the exported zips remain in Drive.


In [ ]:
from datetime import datetime
from pathlib import Path
import zipfile

WORKSPACE_ROOT = Path('/content/drive/MyDrive/direction3_multidataset_workspace')
REPO_DIR = WORKSPACE_ROOT / 'synthetic-data-lifecycle-benchmarks'
EXPORT_DIR = WORKSPACE_ROOT / 'exports'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


def dataset_roots(dataset: str):
    if dataset == 'adult':
        return [
            REPO_DIR / 'results',
            REPO_DIR / 'models' / 'saved',
        ]
    if dataset in {'bank', 'diabetes', 'covertype'}:
        return [REPO_DIR / 'benchmarks' / 'results' / 'dp_triangle' / dataset]
    raise ValueError(f'Unknown dataset: {dataset}')


def archive_dataset(dataset: str) -> Path:
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    archive_path = EXPORT_DIR / f'{dataset}_direction3_{stamp}.zip'
    with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for root in dataset_roots(dataset):
            if not root.exists():
                print('Skipping missing path:', root)
                continue
            for file_path in root.rglob('*'):
                if file_path.is_file():
                    zf.write(file_path, file_path.relative_to(REPO_DIR))
    print('Saved archive:', archive_path)
    return archive_path


def latest_exports():
    return sorted(EXPORT_DIR.glob('*.zip'))


## Full Direction 3 Runs

Run one dataset at a time. Each cell keeps the outputs in Drive and also creates a dataset-specific zip in Drive when the run finishes.


In [ ]:
%cd /content/drive/MyDrive/direction3_multidataset_workspace/synthetic-data-lifecycle-benchmarks
!python -m dp_triangle.run_direction3 --dataset adult --refresh-results
adult_archive = archive_dataset('adult')
adult_archive


In [ ]:
%cd /content/drive/MyDrive/direction3_multidataset_workspace/synthetic-data-lifecycle-benchmarks
!python -m dp_triangle.run_direction3 --dataset bank --refresh-results
bank_archive = archive_dataset('bank')
bank_archive


In [ ]:
%cd /content/drive/MyDrive/direction3_multidataset_workspace/synthetic-data-lifecycle-benchmarks
!python -m dp_triangle.run_direction3 --dataset diabetes --refresh-results
diabetes_archive = archive_dataset('diabetes')
diabetes_archive


In [ ]:
%cd /content/drive/MyDrive/direction3_multidataset_workspace/synthetic-data-lifecycle-benchmarks
!python -m dp_triangle.run_direction3 --dataset covertype --refresh-results
covertype_archive = archive_dataset('covertype')
covertype_archive


## Inspect Dashboards


In [ ]:
import pandas as pd
from pathlib import Path

REPO_DIR = Path('/content/drive/MyDrive/direction3_multidataset_workspace/synthetic-data-lifecycle-benchmarks')

display(pd.read_csv(REPO_DIR / 'results' / 'dp_triangle_dashboard.csv'))
display(pd.read_csv(REPO_DIR / 'benchmarks' / 'results' / 'dp_triangle' / 'bank' / 'dp_triangle_dashboard.csv'))
display(pd.read_csv(REPO_DIR / 'benchmarks' / 'results' / 'dp_triangle' / 'diabetes' / 'dp_triangle_dashboard.csv'))
display(pd.read_csv(REPO_DIR / 'benchmarks' / 'results' / 'dp_triangle' / 'covertype' / 'dp_triangle_dashboard.csv'))


In [ ]:
from IPython.display import Image, display
from pathlib import Path

REPO_DIR = Path('/content/drive/MyDrive/direction3_multidataset_workspace/synthetic-data-lifecycle-benchmarks')
image_paths = [
    REPO_DIR / 'results' / 'figure4_epsilon_tradeoff_curve.png',
    REPO_DIR / 'results' / 'figure5_pff_radar_chart.png',
    REPO_DIR / 'results' / 'figure6_intersectional_fairness.png',
    REPO_DIR / 'results' / 'figure7_post_hoc_recovery.png',
    REPO_DIR / 'benchmarks' / 'results' / 'dp_triangle' / 'bank' / 'figure4_epsilon_tradeoff_curve.png',
    REPO_DIR / 'benchmarks' / 'results' / 'dp_triangle' / 'bank' / 'figure5_pff_radar_chart.png',
    REPO_DIR / 'benchmarks' / 'results' / 'dp_triangle' / 'diabetes' / 'figure4_epsilon_tradeoff_curve.png',
    REPO_DIR / 'benchmarks' / 'results' / 'dp_triangle' / 'diabetes' / 'figure5_pff_radar_chart.png',
    REPO_DIR / 'benchmarks' / 'results' / 'dp_triangle' / 'covertype' / 'figure4_epsilon_tradeoff_curve.png',
]

for image_path in image_paths:
    if image_path.exists():
        print(image_path)
        display(Image(filename=str(image_path)))


## Final Combined Archive

This creates one all-in-one zip in Drive after the per-dataset runs are complete.


In [ ]:
from datetime import datetime
from pathlib import Path
import zipfile

REPO_DIR = Path('/content/drive/MyDrive/direction3_multidataset_workspace/synthetic-data-lifecycle-benchmarks')
EXPORT_DIR = Path('/content/drive/MyDrive/direction3_multidataset_workspace/exports')
combined_archive = EXPORT_DIR / f'direction3_multidataset_outputs_{datetime.now().strftime("%Y%m%d_%H%M%S")}.zip'
roots = [
    REPO_DIR / 'results',
    REPO_DIR / 'models' / 'saved',
    REPO_DIR / 'benchmarks' / 'results' / 'dp_triangle',
]

with zipfile.ZipFile(combined_archive, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for root in roots:
        if not root.exists():
            print('Skipping missing path:', root)
            continue
        for file_path in root.rglob('*'):
            if file_path.is_file():
                zf.write(file_path, file_path.relative_to(REPO_DIR))

print('Combined archive:', combined_archive)
combined_archive


In [ ]:
latest_exports()


## Resume After A Runtime Reset

If Colab disconnects later:
- reconnect the runtime
- remount Drive
- rerun the clone/update cell
- rerun the install cell if needed
- continue from the dataset cell you had not finished yet

Because the repo and outputs live in Drive, previously completed datasets should still be there.
